### parallelization with MPI
on helen

from https://emcee.readthedocs.io/en/stable/tutorials/parallel/

In [1]:
%matplotlib inline

import os 
os.environ["OMP_NUM_THREADS"] = "1" 
# ensures that numpy doesnt automatically parallelize
# certain tasks while i assign workers to do things


In [2]:
import time 
import numpy as np

def log_prob(theta):
    t = time.time() + np.random.uniform(0.005, 0.008)
    while True:
        if time.time() >= t:
            break
    return -0.5*np.sum(theta**2)

In [3]:
import emcee 
np.random.seed(42)
initial = np.random.randn(32, 5) 
nwalkers, ndim = initial.shape 
nsteps = 100

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
start=time.time()
sampler.run_mcmc(initial, nsteps, progress=True)
end=time.time()
serial_time = end-start
print("Serial took {0:.1f} seconds".format(serial_time))

100%|██████████| 100/100 [00:20<00:00,  4.78it/s]

Serial took 21.2 seconds


### multiprocessing 

In [4]:
from multiprocessing import cpu_count

ncpu = cpu_count()
print("{0} CPUs".format(ncpu))

print(ncpu)

128 CPUs
128


In [5]:
mpi_time = !mpiexec -n {ncpu} python script.pympi_time = !mpiexec -n {ncpu} python script.py # {ncpu} throws error
# mpi_time = !mpiexec --use-hwthread-cpus -n 128 python script.py
# mpi_time = float(mpi_time) # not [0] since auth errors
# print("MPI took {0:.1f} seconds".format(mpi_time))
# print("{0:.1f} times faster than serial".format(serial_time / mpi_time))

In [6]:
mpi_time

['Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 '--------------------------------------------------------------------------',
 'There are not enough slots available in the system to satisfy the 128',
 'slots that were requested by the application:',
 '',
 '  python',
 '',
 'Either request fewer slots for your application, or make more slots',
 'available for use.',
 '',
 'A "slot" is the Open MPI term for an allocatable unit where we can',
 'launch a process.  The number of slots available are defined by the',
 'environment in which Open MPI processes are run:',
 '',
 '  1. Hostfile, via "slots=N" clauses (N defaults to number of',
 '     processor cores if not provided)',
 '  2. The --host command line parameter, via a ":N" suffix on the',
 '     hostname (N defaults to 1 if not provided)',
 '  3. Resource manager (e.g., SLURM, PBS/Torque, LSF, etc.)',
 '  4. If none of a hostfile, the -

##### debug

from the tutorial

In [7]:
mpi_time = !mpiexec -n {ncpu} python script.py

number cpus 

In [9]:
ncpu

128

error message for `mpi_time`:
- it seems i dont have enough cpus 
- also for some reason there are many authorization errors no matter how i try to do this

In [10]:
mpi_time 

['Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 '--------------------------------------------------------------------------',
 'There are not enough slots available in the system to satisfy the 128',
 'slots that were requested by the application:',
 '',
 '  python',
 '',
 'Either request fewer slots for your application, or make more slots',
 'available for use.',
 '',
 'A "slot" is the Open MPI term for an allocatable unit where we can',
 'launch a process.  The number of slots available are defined by the',
 'environment in which Open MPI processes are run:',
 '',
 '  1. Hostfile, via "slots=N" clauses (N defaults to number of',
 '     processor cores if not provided)',
 '  2. The --host command line parameter, via a ":N" suffix on the',
 '     hostname (N defaults to 1 if not provided)',
 '  3. Resource manager (e.g., SLURM, PBS/Torque, LSF, etc.)',
 '  4. If none of a hostfile, the -

maximum number of slots:

In [11]:
for n in [64]: # not enough required slots >64
    print(f"\n=== {n} ===") 
    !mpiexec -n {n} hostname


=== 64 ===
Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen
helen


In [12]:
mpi_time = !mpiexec -n 128 python script.py

In [13]:
mpi_time # not enough slots

['Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 '--------------------------------------------------------------------------',
 'There are not enough slots available in the system to satisfy the 128',
 'slots that were requested by the application:',
 '',
 '  python',
 '',
 'Either request fewer slots for your application, or make more slots',
 'available for use.',
 '',
 'A "slot" is the Open MPI term for an allocatable unit where we can',
 'launch a process.  The number of slots available are defined by the',
 'environment in which Open MPI processes are run:',
 '',
 '  1. Hostfile, via "slots=N" clauses (N defaults to number of',
 '     processor cores if not provided)',
 '  2. The --host command line parameter, via a ":N" suffix on the',
 '     hostname (N defaults to 1 if not provided)',
 '  3. Resource manager (e.g., SLURM, PBS/Torque, LSF, etc.)',
 '  4. If none of a hostfile, the -

In [14]:
mpi_time = !mpiexec -n 64 python script.py

In [ ]:
mpi_time # time is 1.6564040184020996 

['Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization protocol specified',
 '',
 'Authorization required, but no authorization prot

other things i tried that run but have a worse runtime (aside from the authorization problem - though i think the time is probably wrong because of this error):

In [ ]:
mpi_output = !mpiexec --oversubscribe -n 128 python script.py 
print("\n".join(mpi_output)) # time is 1.7214796543121338

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, b

using hyperthreads as slots also works, but the performance is even worse

In [ ]:
mpi_time = !mpiexec --use-hwthread-cpus -n 128 python script.py

for i, line in enumerate(mpi_time): # time is 1.8942904472351074
    print(i, repr(line))

0 'Authorization required, but no authorization protocol specified'
1 ''
2 'Authorization required, but no authorization protocol specified'
3 ''
4 'Authorization required, but no authorization protocol specified'
5 ''
6 'Authorization required, but no authorization protocol specified'
7 ''
8 'Authorization required, but no authorization protocol specified'
9 ''
10 'Authorization required, but no authorization protocol specified'
11 ''
12 'Authorization required, but no authorization protocol specified'
13 ''
14 'Authorization required, but no authorization protocol specified'
15 ''
16 'Authorization required, but no authorization protocol specified'
17 ''
18 'Authorization required, but no authorization protocol specified'
19 ''
20 'Authorization required, but no authorization protocol specified'
21 ''
22 'Authorization required, but no authorization protocol specified'
23 ''
24 'Authorization required, but no authorization protocol specified'
25 ''
26 'Authorization required, but no 